# 07 Model selection

Two questions:

1. Which individual models perform best?
2. Does a system that selects between them, or averages them, forecast better than its individual candidates?

Nothing is trained here. We load the forecasts saved by earlier notebooks and choose between them using only evidence that was complete on the decision date.

The live path, sections 1 to 5, uses historical-only candidates and runs in about two minutes. The optional sections after it cover finer hierarchy levels and selection objectives, penalties and winner stability, candidate-pool size, a pool with planned-scenario models, a hindsight bound and an ensemble candidate. They take about 19 more minutes.


In [1]:
import os
import sys
import warnings

if "google.colab" in sys.modules:
    !git clone -q https://github.com/tackes/time-series-forecasting-bootcamp
    %cd time-series-forecasting-bootcamp
    %pip install -q statsforecast==2.1.1 mlforecast==1.1.0 utilsforecast==0.2.17 lightgbm==4.7.0 catboost==1.2.10
elif not os.path.exists("shared"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd

from shared.prepare_data import load_panel, H, DEV_CUTOFFS, TEST_CUTOFF, STABILITY_CUTOFFS, completed_runs
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import bias, mae

from shared.evaluation import common_rows, to_wide, score, apc_blocks
from shared.leaderboard import load_forecasts, save_forecasts, leaderboard, combine_tiers

panel = load_panel()
actuals = panel[["unique_id", "ds", "y"]]

## 1. Candidates

Every historical_only forecast saved in notebooks 01 to 06: the statistical baselines and the LightGBM variants. They include weak ones (raw target, min-max scaling) on purpose, so we can see whether weak candidates hurt or help.

Models that use a promotion plan (planned_scenario), oracle models and the leaky price model are not candidates here. A system that could choose between them inherits the highest tier among them.

In [3]:
files = ["01_baselines", "02_ml_baseline", "03_exogenous", "03b_promotion_spikes", "04_global_vs_specialized", "05_horizon_strategies", "06_objectives"]
forecasts = load_forecasts(files)
forecasts = forecasts[~forecasts.model.isin(["Selected_per_SKU", "Average"])]    # preliminary strategies from notebook 01

tier_of = forecasts.drop_duplicates("model").set_index("model").information_tier
stats = ["Naive", "HistoricAverage", "SeasonalNaive"]
ml = sorted(tier_of[(tier_of == "historical_only")].index.difference(stats))
planned = sorted(tier_of[tier_of == "planned_scenario"].index)

candidates = forecasts[forecasts.model.isin(stats + ml + planned)]
pools = {"stats": stats, "ml": ml, "stats+ml": stats + ml, "stats+ml+planned": stats + ml + planned}
print(f"{len(stats)} statistical, {len(ml)} historical ML, {len(planned)} planned-scenario candidates")

columns = ["model", "coverage", "wape", "bias", "mean_rmsse", "median_rmsse", "apc"]
dev_board = leaderboard(candidates[candidates.model.isin(stats + ml)], actuals, scope="dev")
dev_board[columns].round(3).head(10)

3 statistical, 15 historical ML, 4 planned-scenario candidates


,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc
0,lgbm_l1_std,1.000,0.378,-0.039,1.705,0.833,0.143
1,lgbm_promo_calendar_none,1.000,0.396,-0.062,1.759,0.860,0.139
2,lgbm_promo_none,1.000,0.396,-0.019,1.739,0.848,0.138
3,lgbm_direct,1.000,0.398,-0.033,1.765,0.873,0.125
4,SeasonalNaive,0.988,0.408,-0.042,1.850,1.055,0.000
5,lgbm_l1_raw,1.000,0.409,-0.143,1.757,0.858,0.183
6,HistoricAverage,1.000,0.426,-0.060,1.791,0.887,0.015
7,lgbm_horizon_safe,1.000,0.427,-0.012,1.806,0.907,0.088
8,lgbm_tweedie_raw,1.000,0.432,0.008,1.918,0.951,0.197
9,lgbm_dept_std,1.000,0.440,0.119,1.865,0.918,0.157


## 2. Selection by hand

The simplest tournament: for each SKU, the candidate with the lowest MAE over the seven development runs, then that candidate's forecast at the final cutoff.

`evaluate` with the SKU as the series id gives one MAE per SKU and candidate. That is the selection error. The leaderboard is a different calculation: it pools errors over all SKUs into one WAPE.

In [4]:
evidence_rows = common_rows(candidates[candidates.model.isin(stats + ml) & candidates.cutoff.isin(DEV_CUTOFFS)]).merge(actuals, on=["unique_id", "ds"])
sku_mae = evaluate(to_wide(evidence_rows), metrics=[mae], models=sorted(stats + ml)).drop(columns="metric").set_index("unique_id")
sku_winner = sku_mae.idxmin(axis=1)
sku_mae.iloc[:3, :6].round(1)

,HistoricAverage,Naive,SeasonalNaive,lgbm_baseline,lgbm_dept_raw,lgbm_dept_std
unique_id,,,,,,
10006_S1,646.1,1106.3,572.4,747.9,738.4,746.4
10008_S1,263.8,319.3,356.0,250.2,228.7,301.8
10010_S1,2272.1,714.3,1870.6,1345.8,1984.5,1535.1


In [5]:
sku_winner.value_counts()

SeasonalNaive               315
HistoricAverage             298
lgbm_l1_raw                 245
Naive                       150
lgbm_horizon_safe           115
lgbm_l1_std                  97
lgbm_promo_calendar_none     96
lgbm_global_minmax           95
lgbm_quantile60_std          86
lgbm_promo_none              84
lgbm_direct                  81
lgbm_tweedie_raw             71
lgbm_baseline                50
lgbm_dept_std                49
lgbm_dept_raw                47
lgbm_raw                     44
lgbm_hist_exog               40
lgbm_velocity_std            37
Name: count, dtype: int64

Apply the choice at the final cutoff and score it with the same leaderboard as every other model.

In [6]:
final_rows = candidates[candidates.cutoff == TEST_CUTOFF]
by_hand = final_rows[final_rows.model == final_rows.unique_id.map(sku_winner)].assign(model="By_hand_sku")
leaderboard(pd.concat([final_rows[final_rows.model.isin(["lgbm_l1_std", "SeasonalNaive"])], by_hand]), actuals, scope="test")[columns].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc
0,lgbm_l1_std,1.0,0.369,-0.003,1.265,0.671,NaN
1,By_hand_sku,1.0,0.395,0.039,1.316,0.696,NaN
2,SeasonalNaive,1.0,0.438,-0.037,1.525,0.929,NaN


## 3. Selection at every decision date

The functions below do the same thing at any hierarchy level and at every decision date, using only the runs completed by then. At the final cutoff and SKU level they reproduce the choice above as `Select_all_sku`.

Ties matter more than they look. 84 SKUs tie exactly on development MAE between candidates, and `idxmin` gives a tie to the first column, here the first model alphabetically. Listing the models as `stats + ml` instead (Naive first) changes those 84 choices and moves the final-window WAPE from 0.395 to 0.388.

A selection policy answers: for each series, which candidate's forecast do we use?

- **Level**: the group of series that shares one choice: portfolio (one model for everything), department, class, subclass or SKU.
- **Cost**: mean absolute error of the candidate over the evidence rows in the group, from `utilsforecast` `mae` with the group as the series id.
- **Evidence**: the forecast runs whose whole 14-week horizon was observed on the decision date, from `completed_runs(date)`. For the final cutoff that is all seven development runs. For a development decision on 2025-01-01 it is only the three runs through 2024-09-11.
- **Fallback**: a series with no evidence, or whose chosen model has no forecast at that cutoff, gets the portfolio winner.

In [7]:
hier = panel.drop_duplicates("unique_id").set_index("unique_id")
hier = pd.DataFrame({
    "portfolio": "all",
    "department": hier.department_name.fillna("Unknown"),
    "class": hier.class_id.astype("string").fillna("Unknown"),
    "subclass": hier.subclass_id.astype("string").fillna("Unknown"),
    "sku": hier.index,
}, index=hier.index)

dev_rows = candidates[candidates.cutoff.isin(DEV_CUTOFFS)].merge(actuals, on=["unique_id", "ds"])
evidence = {name: common_rows(dev_rows[dev_rows.model.isin(models)]) for name, models in pools.items()}

In [8]:
def cost_table(pool, runs, level, aggregate="sku", lam=0.0, mu=0.0):
    """Selection cost per group (rows) and candidate (columns), from utilsforecast mae and bias."""
    ev = evidence[pool]
    ev = ev[ev.cutoff.isin(runs)].assign(group=lambda d: d.unique_id.map(hier[level]))
    if aggregate == "hierarchy":
        ev = ev.groupby(["group", "model", "cutoff", "ds"], as_index=False)[["y", "yhat"]].sum().assign(unique_id=lambda d: d.group)
    models = sorted(pools[pool])
    errors = evaluate(to_wide(ev, extra=["group"]), metrics=[mae, bias], models=models, id_col="group").set_index(["metric", "group"])
    cost = errors.loc["mae"] + lam * errors.loc["bias"].abs()
    if mu:
        fcst = candidates[candidates.model.isin(pools[pool]) & candidates.cutoff.isin(runs)]
        blocks = apc_blocks(fcst, list(runs)[1:])
        blocks["group"] = blocks.unique_id.map(hier[level])
        revision = blocks.groupby(["group", "model"])[["change", "base"]].sum()
        revision = (revision.change / revision.base.where(revision.base > 0)).unstack("model")
        mean_y = ev[ev.model == models[0]].groupby("group").y.mean()
        cost += mu * revision.reindex_like(cost).fillna(0).mul(mean_y, axis=0)
    return cost


def choose(pool, runs, level="sku", aggregate="sku", lam=0.0, mu=0.0, margin=0.0, default=None):
    """Model name for every series, and the default used as fallback (the portfolio winner unless given)."""
    cost = cost_table(pool, runs, level, aggregate, lam, mu)
    default = default or cost_table(pool, runs, "portfolio", aggregate, lam, mu).iloc[0].idxmin()
    pick = cost.idxmin(axis=1)
    gain = 1 - cost.min(axis=1) / cost[default]
    pick = pick.where(gain > margin, default)
    return hier[level].map(pick), default


def strategy(name, pool, decisions, picks=None, **policy):
    """Selected forecasts at each decision cutoff, using only the runs completed by then."""
    out = []
    for cutoff in decisions:
        choice, default = choose(pool, completed_runs(cutoff), **policy)
        if picks is not None:
            picks[(name, cutoff)] = choice
        rows = candidates[candidates.model.isin(pools[pool]) & (candidates.cutoff == cutoff)]
        picked = rows[rows.model == rows.unique_id.map(choice)]
        fallback = rows[(rows.model == default) & ~rows.unique_id.isin(picked.unique_id)]
        out.append(pd.concat([picked, fallback]).assign(model=name))
    tier = combine_tiers(tier_of[pools[pool]])
    return pd.concat(out).assign(experiment="07_model_selection", information_tier=tier)[["unique_id", "cutoff", "ds", "model", "yhat", "experiment", "information_tier"]]


def average(name, models, decisions):
    rows = candidates[candidates.model.isin(models) & candidates.cutoff.isin(decisions)]
    tier = combine_tiers(tier_of[models])
    out = rows.groupby(["unique_id", "cutoff", "ds"], as_index=False).yhat.mean()
    return out.assign(model=name, experiment="07_model_selection", information_tier=tier)


def top3(pool, decision):
    runs = candidates[candidates.model.isin(pools[pool]) & candidates.model.isin(ml) & candidates.cutoff.isin(completed_runs(decision))]
    return score(runs, actuals, rmsse=False).sort_values("wape").model.head(3).tolist()

### One decision by hand

What `choose` does for one level and one date, before the fallback and the optional penalties: department-level selection on 2025-01-01, using only the three runs completed by then. Each candidate's cost is its MAE pooled over every row of the department, from `evaluate` with the department as the series id.

In [9]:
decision = pd.Timestamp("2025-01-01")
runs = completed_runs(decision)
rows = evidence["stats+ml"][evidence["stats+ml"].cutoff.isin(runs)].assign(group=lambda d: d.unique_id.map(hier["department"]))
department_mae = evaluate(to_wide(rows, extra=["group"]), metrics=[mae], models=sorted(stats + ml), id_col="group").drop(columns="metric").set_index("group")
department_winner = department_mae.idxmin(axis=1)
department_winner.value_counts().head(5)

lgbm_l1_std         31
lgbm_l1_raw         24
SeasonalNaive       15
lgbm_tweedie_raw    12
lgbm_promo_none      9
Name: count, dtype: int64

`choose` gives the same answer for every SKU, unless a department's winner fails to beat the portfolio winner, when the SKU falls back to it:

In [10]:
by_function, portfolio_winner = choose("stats+ml", runs, level="department")
pd.Series({"departments": len(department_winner), "distinct winners": department_winner.nunique(),
           "SKUs where choose agrees": (by_function == hier["department"].map(department_winner)).mean()})

departments                 126.0
distinct winners             16.0
SKUs where choose agrees      1.0
dtype: float64

## 4. Development comparison

Each development decision uses only runs completed by that date. The decisions are the development cutoffs that have at least one completed run: 2024-11-06, 2024-12-04 and 2025-01-01, with one, two and three evidence runs. Evidence is thin, so differences of a point or two of WAPE here are noise.

Strategies and individual candidates are scored together on the same product-weeks.

In [11]:
decisions = [d for d in DEV_CUTOFFS if completed_runs(d)]
policies = {
    "Select_stats_sku": ("stats", dict(level="sku")),
    "Select_ml_sku": ("ml", dict(level="sku")),
    "Select_all_portfolio": ("stats+ml", dict(level="portfolio")),
    "Select_all_department": ("stats+ml", dict(level="department")),
    "Select_all_sku": ("stats+ml", dict(level="sku")),
}

def build(decisions):
    out = [strategy(name, pool, decisions, **policy) for name, (pool, policy) in policies.items()]
    for d in decisions:
        out.append(average("Average_stats", stats, [d]))
        out.append(average("Average_all", stats + ml, [d]))
        out.append(average("Average_top3_ml", top3("ml", d), [d]))
    return pd.concat(out)

dev_strategies = build(decisions)
individual = candidates[candidates.model.isin(stats + ml)]
leaderboard(pd.concat([individual, dev_strategies]), actuals, scope=decisions)[columns].round(3).head(14)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc
0,Select_all_sku,1.000,0.366,-0.002,1.486,0.833,0.188
1,Average_stats,1.000,0.366,-0.059,1.431,0.814,0.140
2,Select_stats_sku,1.000,0.379,-0.044,1.514,0.852,0.168
3,SeasonalNaive,0.997,0.380,-0.056,1.601,1.011,0.000
4,Select_all_portfolio,1.000,0.385,-0.007,1.484,0.809,0.199
5,lgbm_l1_std,1.000,0.385,-0.007,1.484,0.809,0.199
6,Average_top3_ml,1.000,0.386,-0.043,1.493,0.801,0.193
7,lgbm_promo_none,1.000,0.391,-0.031,1.485,0.814,0.172
8,Select_ml_sku,1.000,0.391,0.008,1.521,0.838,0.206
9,Average_all,1.000,0.392,0.042,1.757,0.885,0.145


## 5. Final 14 weeks

Policies are fixed above and applied once at the final cutoff with all seven development runs as evidence. To measure stability the same policies are also applied at 2025-02-12 and 2025-03-12, each with only the runs completed by then.

This window was inspected while building these notebooks, so treat the numbers as an exploratory benchmark rather than an untouched holdout.

In [12]:
targets = [TEST_CUTOFF] + list(STABILITY_CUTOFFS)
strategies = build(targets)

dev_best = dev_board.model.iloc[0]
members = stats + [dev_best, "lgbm_baseline"]
test_board = leaderboard(pd.concat([individual[individual.model.isin(members)], strategies]), actuals, scope="test")
test_board[columns + ["information_tier"]].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc,information_tier
0,Average_top3_ml,1.0,0.364,-0.050,1.266,0.679,0.099,historical_only
1,Select_all_portfolio,1.0,0.369,-0.003,1.265,0.671,0.118,historical_only
2,lgbm_l1_std,1.0,0.369,-0.003,1.265,0.671,0.118,historical_only
3,Select_ml_sku,1.0,0.373,0.002,1.297,0.695,0.151,historical_only
4,Select_all_department,1.0,0.376,-0.004,1.279,0.692,0.133,historical_only
5,Average_all,1.0,0.378,0.063,1.900,0.757,0.092,historical_only
6,Average_stats,1.0,0.384,0.006,1.275,0.726,0.109,historical_only
7,Select_all_sku,1.0,0.395,0.039,1.316,0.696,0.114,historical_only
8,lgbm_baseline,1.0,0.419,0.109,1.302,0.733,0.092,historical_only
9,Select_stats_sku,1.0,0.425,0.029,1.371,0.737,0.090,historical_only


### Does selection improve on its candidates?

Each system against the best individual candidate by development WAPE (`dev_best`, chosen without the final window) and against the stats-only system. Negative changes are improvements.

In [13]:
board = test_board.set_index("model")
rank = leaderboard(individual, actuals, scope="test").set_index("model")
rows = ["Select_stats_sku", "Select_ml_sku", "Select_all_portfolio", "Select_all_department", "Select_all_sku", "Average_stats", "Average_all", "Average_top3_ml"]
metrics = ["wape", "bias", "mean_rmsse", "apc"]
impact = board.loc[rows, metrics]
pd.concat({
    "level": impact,
    "change vs dev_best": impact - board.loc[dev_best, metrics],
    "change vs stats-only": impact - board.loc["Select_stats_sku", metrics],
}, axis=1).round(3)

level                          change vs dev_best  \
                        wape   bias mean_rmsse    apc               wape   
model                                                                      
Select_stats_sku       0.425  0.029      1.371  0.090           0.056459   
Select_ml_sku          0.373  0.002      1.297  0.151           0.004063   
Select_all_portfolio   0.369 -0.003      1.265  0.118                0.0   
Select_all_department  0.376 -0.004      1.279  0.133           0.007647   
Select_all_sku         0.395  0.039      1.316  0.114            0.02605   
Average_stats          0.384  0.006      1.275  0.109           0.015016   
Average_all            0.378  0.063      1.900  0.092           0.009001   
Average_top3_ml        0.364 -0.050      1.266  0.099          -0.004411   

                                                     change vs stats-only  \
                           bias mean_rmsse       apc                 wape   
model                                                                       
Select_stats_sku       0.032046    0.10608 -0.028085                  0.0   
Select_ml_sku          0.005233   0.032326  0.032651            -0.052396   
Select_all_portfolio        0.0        0.0       0.0            -0.056459   
Select_all_department -0.000725   0.013898    0.0142            -0.048812   
Select_all_sku         0.041534   0.051247 -0.003936             -0.03041   
Average_stats          0.008715   0.009725 -0.009183            -0.041443   
Average_all            0.065899   0.634751 -0.026741            -0.047459   
Average_top3_ml       -0.047181   0.000979 -0.019066             -0.06087   

                                                      
                           bias mean_rmsse       apc  
model                                                 
Select_stats_sku            0.0        0.0       0.0  
Select_ml_sku         -0.026813  -0.073754  0.060736  
Select_all_portfolio  -0.032046   -0.10608  0.028085  
Select_all_department  -0.03277  -0.092182  0.042285  
Select_all_sku         0.009489  -0.054833  0.024149  
Average_stats         -0.023331  -0.096356  0.018902  
Average_all            0.033853   0.528671  0.001344  
Average_top3_ml       -0.079226  -0.105101  0.009019

For context, where the strategies and `dev_best` fall among all historical-only individual candidates on the same window:

In [14]:
rank[["wape", "bias", "mean_rmsse", "apc"]].round(3).head(8)

,wape,bias,mean_rmsse,apc
model,,,,
lgbm_promo_calendar_none,0.367,-0.101,1.283,0.100
lgbm_l1_std,0.369,-0.003,1.265,0.118
lgbm_promo_none,0.374,-0.047,1.272,0.093
lgbm_l1_raw,0.375,-0.073,1.229,0.166
lgbm_direct,0.380,-0.033,1.301,0.128
lgbm_dept_std,0.403,0.079,1.312,0.133
lgbm_horizon_safe,0.416,-0.061,1.378,0.033
lgbm_hist_exog,0.418,0.119,1.297,0.094


### Reading the result

- One model for the whole portfolio, chosen on development evidence, is `lgbm_l1_std`: 0.369 WAPE in the final window.
- Choosing per SKU looked best on the development decisions (0.366) and is worse in the final window (0.395). With a few runs per SKU, much of the choice is noise.
- The average of the top three ML models (0.364) is the best historical-only result. The average of all 18 candidates (0.378) is not.
- The optional sections test this further with finer hierarchy levels, larger candidate pools and an ensemble candidate. None beats the top-three average once the evidence covers several runs.

## Optional: finer levels, penalties and stability

These sweeps are for exploring the policy, not for the live session. They use the same time-respecting development decisions.

### Selection level and objective across the merchandise hierarchy

Five levels: portfolio, department, class, subclass and SKU (`unique_id`, a SKU at a store). A SKU with a missing hierarchy value goes into an `Unknown` group at that level; three SKUs are affected.

Three objectives choose the winning candidate in each group:

- **pooled**: mean absolute error of the SKU forecasts, pooled over every SKU-week in the group.
- **aggregated**: actuals and forecasts are first summed to group-week, then mean absolute error. Identical to pooled for single-SKU groups, so it is skipped at SKU level.
- **mae_bias**: pooled mean absolute error plus the absolute mean error, the penalty used in the optional sweep below (`lam=1`).

All strategies use the same `stats+ml` candidates, evidence rules and scored product-weeks. The pooled portfolio, department and SKU policies are the live-path strategies above and keep their names.

In [15]:
levels = ["portfolio", "department", "class", "subclass", "sku"]
objectives = {"pooled": {}, "aggregated": dict(aggregate="hierarchy"), "mae_bias": dict(lam=1.0)}
live = {"portfolio": "Select_all_portfolio", "department": "Select_all_department", "sku": "Select_all_sku"}

grid = {}
for level in levels:
    for objective, options in objectives.items():
        if objective == "aggregated" and level == "sku":
            continue
        name = live.get(level, f"Select_{level}_pooled") if objective == "pooled" else f"Select_{level}_{objective}"
        grid[name] = ("stats+ml", dict(level=level, **options))

new_policies = {name: spec for name, spec in grid.items() if name not in policies}
hier_runs = pd.concat([strategy(name, pool, decisions + targets, **policy) for name, (pool, policy) in new_policies.items()])
grid_runs = pd.concat([dev_strategies[dev_strategies.model.isin(grid)], strategies[strategies.model.isin(grid)], hier_runs])

### Final 14 weeks

Each group is chosen with all seven development runs and applied at the final cutoff. Groups and distinct models are counted for that decision; the model count includes the portfolio fallback.

In [16]:
choices = {name: choose(pool, completed_runs(TEST_CUTOFF), **policy)[0] for name, (pool, policy) in grid.items()}
sizes = pd.DataFrame({name: {"n_groups": hier[policy["level"]].nunique(), "n_models": choices[name].nunique()} for name, (_, policy) in grid.items()}).T

hier_test = leaderboard(pd.concat([individual[individual.model.isin(members)], grid_runs]), actuals, scope="test").set_index("model")
hier_test.loc[list(grid), ["wape", "bias", "mean_rmsse", "median_rmsse", "apc", "coverage"]].join(sizes).sort_values("wape").round(3)

,wape,bias,mean_rmsse,median_rmsse,apc,coverage,n_groups,n_models
model,,,,,,,,
Select_all_portfolio,0.369,-0.003,1.265,0.671,0.118,1.0,1,1
Select_class_pooled,0.370,-0.004,1.295,0.699,0.115,1.0,345,18
Select_portfolio_mae_bias,0.374,-0.047,1.272,0.681,0.093,1.0,1,1
Select_all_department,0.376,-0.004,1.279,0.692,0.133,1.0,126,17
Select_class_mae_bias,0.380,0.038,1.323,0.728,0.153,1.0,345,18
Select_subclass_pooled,0.385,0.025,1.447,0.717,0.121,1.0,773,18
Select_subclass_mae_bias,0.387,0.067,1.426,0.725,0.143,1.0,773,18
Select_department_mae_bias,0.390,0.019,1.305,0.716,0.134,1.0,126,17
Select_subclass_aggregated,0.392,0.036,1.485,0.738,0.116,1.0,773,18


### Is the ranking stable?

The same strategies at every decision date that has evidence: the three development decisions (one to three evidence runs) and the three later cutoffs (final window and the two stability cutoffs, with four to seven runs). WAPE per cutoff, then the rank of each strategy within the cutoff.

In [17]:
by_cutoff = pd.concat([score(grid_runs[grid_runs.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in decisions + targets])
wape = by_cutoff.pivot(index="model", columns="cutoff", values="wape")
ranks = wape.rank()
stability = pd.DataFrame({"mean_wape": wape.mean(axis=1), "mean_rank": ranks.mean(axis=1), "best_rank": ranks.min(axis=1), "worst_rank": ranks.max(axis=1),
                          "wape_final": wape[TEST_CUTOFF], "rank_final": ranks[TEST_CUTOFF]})
stability.sort_values("mean_rank").round(3)

,mean_wape,mean_rank,best_rank,worst_rank,wape_final,rank_final
model,,,,,,
Select_all_portfolio,0.378,3.250,1.0,9.5,0.369,1.0
Select_subclass_pooled,0.376,3.500,2.0,6.0,0.385,6.0
Select_class_pooled,0.379,4.333,2.0,8.0,0.370,2.0
Select_all_sku,0.378,4.667,1.0,10.0,0.395,10.0
Select_sku_mae_bias,0.387,6.167,1.0,12.0,0.403,12.0
Select_portfolio_mae_bias,0.387,6.583,3.0,12.0,0.374,3.0
Select_all_department,0.387,7.500,2.0,13.0,0.376,4.0
Select_subclass_aggregated,0.386,7.667,5.0,12.0,0.392,9.0
Select_subclass_mae_bias,0.390,8.167,4.0,12.0,0.387,7.0


Change in WAPE against the pooled portfolio policy at each cutoff. Negative means the finer or different policy forecasts better.

In [18]:
(wape.sub(wape.loc["Select_all_portfolio"], axis=1) * 100).round(2).loc[list(grid)].assign(wins=lambda d: (d < 0).sum(axis=1))

cutoff,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-02-12 00:00:00,2025-03-12 00:00:00,2025-04-09 00:00:00,wins
model,,,,,,,
Select_all_portfolio,0.00,0.00,0.00,0.00,0.00,0.00,0
Select_portfolio_aggregated,0.00,1.75,6.23,9.16,7.33,6.89,0
Select_portfolio_mae_bias,0.00,0.00,2.77,1.35,1.00,0.54,0
Select_all_department,1.79,0.20,1.43,0.12,1.37,0.76,0
Select_department_aggregated,0.12,-1.92,3.11,3.98,3.52,3.59,1
Select_department_mae_bias,1.98,0.06,2.02,4.03,3.79,2.08,0
Select_class_pooled,0.26,-1.22,0.03,0.80,0.45,0.14,1
Select_class_aggregated,1.25,-0.80,1.91,2.77,2.94,2.83,1
Select_class_mae_bias,1.04,0.24,0.38,1.99,3.86,1.13,0


### Reading the hierarchy results

- Pooled portfolio selection is best on the final window (WAPE 0.369) and has the best mean rank across the six decision dates. Class pooled is within 0.001 WAPE of it; department, subclass and SKU are 0.007 to 0.026 worse.
- Finer levels gained only on the earliest decisions, when one or two evidence runs left the portfolio winner poorly identified: subclass and SKU beat the portfolio at 2024-11-06 (SKU) and 2024-12-04 (both). From 2025-01-01, with three or more evidence runs, no finer level beats the portfolio by more than 0.01 WAPE points at any cutoff.
- Aggregated MAE is worse than pooled MAE at every level, by 0.7 to 6.9 WAPE points on the final window. Errors of summed forecasts reward models that are right in total and hide the SKU-level errors that WAPE counts.
- The bias penalty (`mae_bias`) does not help at any level. It raises WAPE at every level and does not reduce bias.
- SKU selection uses 18 of the candidates but its SKU winner changes in most runs (see the stability check below), so the choice follows noise.
- Six overlapping windows are not independent evidence, and the final window was inspected during development. Differences of one WAPE point or less are not reliable.

None of the finer selection levels improves the system once evidence is adequate.

### Bias penalty, margin and stability penalty

- `lam` weights the absolute mean error against the mean absolute error (`cost = MAE + lam * |mean error|`).
- `margin`: a group leaves the portfolio winner only if its best candidate lowers the cost by at least this fraction.
- `mu` adds the candidate's own revision size between consecutive evidence runs, converted to units of demand (`mu * APC * mean(actual)`).

In [19]:
variants = {f"lam={lam}": ("stats+ml", dict(level="sku", lam=lam)) for lam in [0, 1, 4]}
variants |= {f"margin={m}": ("stats+ml", dict(level="sku", margin=m)) for m in [0.05, 0.2]}
variants |= {f"mu={mu}": ("stats+ml", dict(level="sku", mu=mu)) for mu in [1, 4]}
runs = pd.concat([strategy(name, pool, decisions, **policy) for name, (pool, policy) in variants.items()])
leaderboard(runs, actuals, scope=decisions)[columns + ["apc_median"]].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc,apc_median
0,margin=0.2,1.0,0.363,-0.009,1.462,0.813,0.194,0.202
1,margin=0.05,1.0,0.364,-0.003,1.481,0.830,0.194,0.184
2,mu=1,1.0,0.365,-0.026,1.506,0.840,0.185,0.145
3,lam=0,1.0,0.366,-0.002,1.486,0.833,0.194,0.179
4,lam=1,1.0,0.373,-0.004,1.489,0.837,0.200,0.195
5,mu=4,1.0,0.374,-0.047,1.522,0.863,0.194,0.149
6,lam=4,1.0,0.394,0.019,1.507,0.852,0.231,0.202


### How stable are the winners?

The best candidate for each SKU in each development run, and how often it changes between consecutive runs.

In [20]:
per_run = evidence["stats+ml"].assign(abs_error=lambda d: (d.yhat - d.y).abs()).groupby(["unique_id", "cutoff", "model"]).abs_error.mean().unstack("model")
winner = per_run.idxmin(axis=1).unstack("cutoff")
changed = (winner.iloc[:, 1:].to_numpy() != winner.iloc[:, :-1].to_numpy()).mean()
print(f"SKU winner differs between consecutive development runs for {changed:.0%} of SKU-run pairs")
print(f"Distinct winners per SKU across {winner.shape[1]} runs: mean {winner.nunique(axis=1).mean():.1f}")

SKU winner differs between consecutive development runs for 71% of SKU-run pairs
Distinct winners per SKU across 7 runs: mean 4.5


## Optional: growing the candidate pool

How does tournament performance change as the pool grows? Five fixed, nested pools of historical-only forecasts already saved by notebooks 01 to 06. Each pool contains the previous one. Nothing is retrained. No planned-scenario, oracle or leaky model is included, so every strategy here is historical_only.

| Pool | Adds |
| --- | --- |
| P1_stats | Naive, HistoricAverage, SeasonalNaive |
| P2_basic | basic LightGBM from notebook 02: raw and standardized target |
| P3_scaled_direct | min-max scaling (04) and direct and horizon-safe forecasting (05) |
| P4_hist_features | historical promotion and calendar features (03) |
| P5_all_hist | everything else: department and velocity models (04), other objectives (06) |

P5 is the `stats+ml` pool used above. For each pool, selection happens at the five hierarchy levels with pooled SKU-level MAE, using the same evidence rules as before.

In [21]:
added = {
    "P1_stats": stats,
    "P2_basic": ["lgbm_raw", "lgbm_baseline"],
    "P3_scaled_direct": ["lgbm_global_minmax", "lgbm_direct", "lgbm_horizon_safe"],
    "P4_hist_features": ["lgbm_hist_exog", "lgbm_promo_none", "lgbm_promo_calendar_none"],
}
added["P5_all_hist"] = [m for m in stats + ml if m not in sum(added.values(), [])]

nested = {}
for pool, models in added.items():
    nested[pool] = sum(list(added.values())[:len(nested) + 1], [])

pools |= nested
evidence |= {pool: evidence["stats+ml"] if pool == "P5_all_hist" else common_rows(dev_rows[dev_rows.model.isin(models)]) for pool, models in nested.items()}

experiment_of = forecasts.drop_duplicates("model").set_index("model").experiment
membership = pd.DataFrame([(pool, m) for pool, models in added.items() for m in models], columns=["pool", "model"])
membership["experiment"] = membership.model.map(experiment_of)
membership["information_tier"] = membership.model.map(tier_of)
membership["pool_size"] = membership.pool.map({pool: len(models) for pool, models in nested.items()})
membership

,pool,model,experiment,information_tier,pool_size
0,P1_stats,Naive,01_baselines,historical_only,3
1,P1_stats,HistoricAverage,01_baselines,historical_only,3
2,P1_stats,SeasonalNaive,01_baselines,historical_only,3
3,P2_basic,lgbm_raw,02_ml_baseline,historical_only,5
4,P2_basic,lgbm_baseline,02_ml_baseline,historical_only,5
5,P3_scaled_direct,lgbm_global_minmax,04_global_vs_specialized,historical_only,8
6,P3_scaled_direct,lgbm_direct,05_horizon_strategies,historical_only,8
7,P3_scaled_direct,lgbm_horizon_safe,05_horizon_strategies,historical_only,8
8,P4_hist_features,lgbm_hist_exog,03_exogenous,historical_only,11
9,P4_hist_features,lgbm_promo_none,03_exogenous,historical_only,11


### Strategies

For every pool and level, the tournament picks each group's winner from the runs completed on the decision date. Two references use the same pool and evidence:

- **Best individual**: the single pool member with the lowest WAPE on the completed development runs.
- **Average**: the simple mean of all pool members.

Existing strategies keep their names: `Select_stats_sku`, the `Select_all_*` and pooled class and subclass strategies for P5, and `Average_stats` and `Average_all`.

In [22]:
reuse = {("P1_stats", "sku"): "Select_stats_sku", ("P5_all_hist", "portfolio"): "Select_all_portfolio", ("P5_all_hist", "department"): "Select_all_department",
         ("P5_all_hist", "class"): "Select_class_pooled", ("P5_all_hist", "subclass"): "Select_subclass_pooled", ("P5_all_hist", "sku"): "Select_all_sku"}
avg_of = {pool: {"P1_stats": "Average_stats", "P5_all_hist": "Average_all"}.get(pool, f"Average_{pool}") for pool in nested}
all_decisions = decisions + targets

tour = pd.DataFrame([(reuse.get((pool, level), f"Tour_{pool}_{level}"), pool, level) for pool in nested for level in levels], columns=["model", "pool", "level"])
new = tour[~tour.model.isin(list(grid) + list(policies))]
tour_runs = pd.concat([strategy(row.model, row.pool, all_decisions, level=row.level) for row in new.itertuples()])
pool_averages = pd.concat([average(avg_of[pool], nested[pool], all_decisions) for pool in ["P2_basic", "P3_scaled_direct", "P4_hist_features"]])


def best_individual(pool, decision):
    runs = candidates[candidates.model.isin(nested[pool]) & candidates.cutoff.isin(completed_runs(decision))]
    name = score(runs, actuals, rmsse=False).sort_values("wape").model.iloc[0]
    rows = candidates[(candidates.model == name) & (candidates.cutoff == decision)]
    return rows.assign(model=f"Best_{pool}", experiment="07_model_selection")[["unique_id", "cutoff", "ds", "model", "yhat", "experiment", "information_tier"]], name


best = {(pool, d): best_individual(pool, d) for pool in nested for d in all_decisions}
best_runs = pd.concat([rows for rows, name in best.values()])

pool_runs = pd.concat([dev_strategies, strategies, grid_runs, tour_runs, pool_averages, best_runs])
pool_runs = pool_runs[pool_runs.model.isin(list(tour.model) + list(avg_of.values()) + [f"Best_{pool}" for pool in nested])].drop_duplicates(["unique_id", "cutoff", "ds", "model"])

### Final 14 weeks (exploratory)

Each matrix has pools as rows and selection levels as columns, scored on the same product-weeks with `leaderboard`. This window was inspected during development, so read it as exploratory. WAPE first.

In [23]:
def matrix(values):
    """Pool x level table from a Series indexed by strategy name."""
    return tour.assign(value=tour.model.map(values)).pivot(index="pool", columns="level", values="value").loc[list(nested), levels]


pool_board = leaderboard(pool_runs, actuals, scope="test").set_index("model")
references = pd.DataFrame({
    "candidates": {pool: len(nested[pool]) for pool in nested},
    "best_individual": {pool: best[(pool, TEST_CUTOFF)][1] for pool in nested},
    "best_wape": {pool: pool_board.wape[f"Best_{pool}"] for pool in nested},
    "average_wape": {pool: pool_board.wape[avg_of[pool]] for pool in nested},
})
display(references.round(3))
matrix(pool_board.wape).round(3)

,candidates,best_individual,best_wape,average_wape
P1_stats,3,SeasonalNaive,0.438,0.384
P2_basic,5,SeasonalNaive,0.438,0.383
P3_scaled_direct,8,lgbm_direct,0.380,0.390
P4_hist_features,11,lgbm_promo_calendar_none,0.367,0.383
P5_all_hist,18,lgbm_l1_std,0.369,0.378


level,portfolio,department,class,subclass,sku
pool,,,,,
P1_stats,0.438,0.430,0.416,0.422,0.425
P2_basic,0.438,0.434,0.417,0.429,0.425
P3_scaled_direct,0.380,0.425,0.423,0.430,0.417
P4_hist_features,0.367,0.406,0.406,0.420,0.409
P5_all_hist,0.369,0.376,0.370,0.385,0.395


Change in WAPE points against the best individual model and against the pool average. Negative means the tournament forecasts better.

In [24]:
vs_best = matrix(pool_board.wape).sub(references.best_wape, axis=0) * 100
vs_average = matrix(pool_board.wape).sub(references.average_wape, axis=0) * 100
pd.concat({"vs best individual": vs_best, "vs pool average": vs_average}, axis=1).round(2)

vs best individual                                  \
level                     portfolio department class subclass   sku   
pool                                                                  
P1_stats                        0.0      -0.75 -2.14    -1.57 -1.24   
P2_basic                        0.0      -0.39 -2.08    -0.87 -1.26   
P3_scaled_direct                0.0       4.48  4.26     4.99  3.71   
P4_hist_features                0.0       3.87  3.93     5.31  4.17   
P5_all_hist                     0.0       0.76  0.14     1.60  2.60   

                 vs pool average                                  
level                  portfolio department class subclass   sku  
pool                                                              
P1_stats                    5.39       4.63  3.25     3.82  4.14  
P2_basic                    5.45       5.06  3.37     4.58  4.19  
P3_scaled_direct           -0.94       3.54  3.32     4.05  2.77  
P4_hist_features           -1.56       2.31  2.37     3.75  2.61  
P5_all_hist                -0.90      -0.14 -0.76     0.70  1.70

Bias, mean RMSSE, APC and forecast coverage, from the same leaderboard.

In [25]:
pd.concat({metric: matrix(pool_board[metric]) for metric in ["bias", "mean_rmsse", "apc", "coverage"]}, axis=1).round(3)

bias                                   mean_rmsse  \
level            portfolio department  class subclass    sku  portfolio   
pool                                                                      
P1_stats            -0.037      0.007  0.026    0.045  0.029      1.525   
P2_basic            -0.037      0.055  0.081    0.101  0.091      1.525   
P3_scaled_direct    -0.033      0.047  0.083    0.098  0.078      1.301   
P4_hist_features    -0.101      0.029  0.065    0.089  0.067      1.283   
P5_all_hist         -0.003     -0.004 -0.004    0.025  0.039      1.265   

                                                         apc             \
level            department  class subclass    sku portfolio department   
pool                                                                      
P1_stats              1.389  1.378    1.376  1.371     0.000      0.085   
P2_basic              1.404  1.463    1.483  1.346     0.000      0.082   
P3_scaled_direct      1.362  1.377    1.480  1.333     0.131      0.093   
P4_hist_features      1.328  1.330    1.465  1.326     0.136      0.093   
P5_all_hist           1.279  1.295    1.447  1.316     0.119      0.133   

                                         coverage                            \
level             class subclass    sku portfolio department class subclass   
pool                                                                          
P1_stats          0.078    0.100  0.092       1.0        1.0   1.0      1.0   
P2_basic          0.097    0.112  0.111       1.0        1.0   1.0      1.0   
P3_scaled_direct  0.103    0.112  0.108       1.0        1.0   1.0      1.0   
P4_hist_features  0.106    0.115  0.109       1.0        1.0   1.0      1.0   
P5_all_hist       0.116    0.121  0.116       1.0        1.0   1.0      1.0   

                       
level             sku  
pool                   
P1_stats          1.0  
P2_basic          1.0  
P3_scaled_direct  1.0  
P4_hist_features  1.0  
P5_all_hist       1.0

### Candidates and distinct winners

Distinct models chosen at each level by the final decision (including the portfolio fallback), and the number of groups at that level.

In [26]:
chosen = {row.model: choose(row.pool, completed_runs(TEST_CUTOFF), level=row.level)[0] for row in tour.itertuples()}
winners = matrix(pd.Series({model: choice.nunique() for model, choice in chosen.items()}))
groups = pd.DataFrame({"groups": {level: hier[level].nunique() for level in levels}}).T
pd.concat([winners.assign(candidates=[len(nested[pool]) for pool in nested]), groups.assign(candidates="")])

,portfolio,department,class,subclass,sku,candidates
P1_stats,1,3,3,3,3,3
P2_basic,1,5,5,5,5,5
P3_scaled_direct,1,8,8,8,8,8
P4_hist_features,1,11,11,11,11,11
P5_all_hist,1,17,18,18,18,18
groups,1,126,345,773,2000,


### Consistency across decision dates

The same strategies at all six decision dates: three development decisions with one to three evidence runs, and the final and two stability cutoffs with four to seven. Counts are out of six dates. The change in WAPE points against the best individual is averaged separately over the development decisions and the later cutoffs.

In [27]:
by_cutoff = pd.concat([score(pool_runs[pool_runs.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in all_decisions])
pool_wape = by_cutoff.pivot(index="model", columns="cutoff", values="wape")
best_wape = pool_wape.loc[[f"Best_{pool}" for pool in tour.pool]].set_axis(tour.model)
average_wape = pool_wape.loc[[avg_of[pool] for pool in tour.pool]].set_axis(tour.model)
delta = pool_wape.loc[tour.model] - best_wape

pd.concat({
    "beats best individual": matrix((pool_wape.loc[tour.model] < best_wape).sum(axis=1)),
    "beats pool average": matrix((pool_wape.loc[tour.model] < average_wape).sum(axis=1)),
    "change vs best, development": matrix(delta[decisions].mean(axis=1) * 100).round(2),
    "change vs best, later cutoffs": matrix(delta[targets].mean(axis=1) * 100).round(2),
}, axis=1)

beats best individual                                \
level                        portfolio department class subclass sku   
pool                                                                   
P1_stats                             0          6     5        6   6   
P2_basic                             0          5     5        6   6   
P3_scaled_direct                     0          0     1        4   4   
P4_hist_features                     0          0     0        4   4   
P5_all_hist                          0          0     1        2   2   

                 beats pool average                                \
level                     portfolio department class subclass sku   
pool                                                                
P1_stats                          0          0     0        0   0   
P2_basic                          0          0     0        0   2   
P3_scaled_direct                  2          0     0        2   4   
P4_hist_features                  3          0     0        4   4   
P5_all_hist                       5          2     6        5   4   

                 change vs best, development                                  \
level                              portfolio department class subclass   sku   
pool                                                                           
P1_stats                                 0.0      -2.96 -2.58    -3.37 -4.64   
P2_basic                                 0.0      -0.64 -2.14    -2.96 -4.71   
P3_scaled_direct                         0.0       1.98  0.80    -0.52 -2.04   
P4_hist_features                         0.0       1.25  0.46    -0.85 -2.18   
P5_all_hist                              0.0       1.14 -0.31    -1.38 -1.88   

                 change vs best, later cutoffs                                  
level                                portfolio department class subclass   sku  
pool                                                                            
P1_stats                                   0.0      -2.41 -4.18    -4.10 -3.74  
P2_basic                                   0.0      -2.86 -4.26    -4.10 -3.88  
P3_scaled_direct                           0.0       2.10  1.80     1.58  1.18  
P4_hist_features                           0.0       1.60  1.49     1.57  1.34  
P5_all_hist                                0.0       0.75  0.46     0.88  1.84

## Optional: a pool that includes planned-scenario models

Adding the planned-scenario promotion models gives the selection more candidates, but the system then needs a promotion plan, so it is labeled planned_scenario. It is a separate comparison from the historical-only system above.

In [28]:
planned_policies = {
    "Select_planned_portfolio": ("stats+ml+planned", dict(level="portfolio")),
    "Select_planned_department": ("stats+ml+planned", dict(level="department")),
    "Select_planned_sku": ("stats+ml+planned", dict(level="sku")),
}
planned_runs = pd.concat([strategy(name, pool, targets, **policy) for name, (pool, policy) in planned_policies.items()]
                         + [average("Average_planned_all", stats + ml + planned, targets)])
planned_members = ["lgbm_baseline", dev_best] + planned[:3]
leaderboard(pd.concat([candidates[candidates.model.isin(planned_members)], planned_runs]), actuals, scope="test")[columns + ["information_tier"]].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc,information_tier
0,lgbm_l1_std,1.0,0.369,-0.003,1.265,0.671,0.122,historical_only
1,lgbm_baseline,1.0,0.419,0.109,1.302,0.733,0.095,historical_only
2,lgbm_promo_calendar_planned,1.0,0.360,-0.060,1.265,0.667,0.185,planned_scenario
3,Select_planned_portfolio,1.0,0.369,-0.003,1.265,0.671,0.122,planned_scenario
4,Average_planned_all,1.0,0.369,0.040,1.751,0.734,0.097,planned_scenario
5,lgbm_adj_calendar_planned,1.0,0.371,-0.102,1.298,0.696,0.180,planned_scenario
6,lgbm_adj_planned,1.0,0.372,-0.112,1.300,0.697,0.181,planned_scenario
7,Select_planned_department,1.0,0.373,-0.014,1.283,0.678,0.144,planned_scenario
8,Select_planned_sku,1.0,0.375,0.016,1.302,0.686,0.135,planned_scenario


## Optional: hindsight bound

The best candidate per SKU chosen with the final-window actuals. It is not achievable, and is labeled leaky_diagnostic.

In [29]:
final = to_wide(candidates[candidates.cutoff == TEST_CUTOFF].merge(actuals, on=["unique_id", "ds"]))
sku_mae = evaluate(final, metrics=[mae], models=sorted(candidates.model.unique())).set_index("unique_id")
best = sku_mae.drop(columns="metric").idxmin(axis=1)
rows = candidates[candidates.cutoff == TEST_CUTOFF]
hindsight = rows[rows.model == rows.unique_id.map(best)].assign(model="Best_in_hindsight_sku", experiment="07_model_selection", information_tier="leaky_diagnostic")

## Optional: can selection improve on a strong ensemble?

`Average_top3_ml` averages the three best ML models on the completed development runs. Here it becomes one more candidate (`Ens_top3`) next to the 18 historical-only models, and the tournaments run again at each hierarchy level.

Membership is chosen on the decision date from completed runs only. The ensemble also needs forecasts for the evidence runs, and a membership picked on the same runs would flatter it. So for each evidence run the members are the top three on the other completed runs (leave one out). The other candidates are unchanged.

Two policies at each level:

- **Hybrid**: pooled MAE tournament over the 19 candidates, the usual rule.
- **Hybrid_margin**: the ensemble is the default. A group switches to an individual model only if that model's cost is at least 5% lower.

Leave-one-out needs two completed runs, so the hybrid starts at the 2024-12-04 decision. The first decision (2024-11-06, one run) is excluded for every strategy compared below.

In [30]:
def top3_of(runs):
    rows = candidates[candidates.model.isin(ml) & candidates.cutoff.isin(runs)]
    return score(rows, actuals, rmsse=False).sort_values("wape").model.head(3).tolist()


def ensemble_evidence(decision):
    """Ensemble forecasts for each completed run, with members chosen on the other completed runs."""
    runs = completed_runs(decision)
    parts = []
    for run in runs:
        rows = candidates[candidates.model.isin(top3_of([r for r in runs if r != run])) & (candidates.cutoff == run)]
        parts.append(rows.groupby(["unique_id", "cutoff", "ds"], as_index=False).yhat.mean().assign(model="Ens_top3"))
    return pd.concat(parts).merge(actuals, on=["unique_id", "ds"])


hybrid_decisions = [d for d in all_decisions if len(completed_runs(d)) > 1]
tier_of["Ens_top3"] = "historical_only"
pools["hybrid"] = stats + ml + ["Ens_top3"]
candidates = pd.concat([candidates] + [average("Ens_top3", top3_of(completed_runs(d)), [d]) for d in hybrid_decisions])

hybrid_policies = {f"Hybrid_{level}": dict(level=level) for level in levels}
hybrid_policies |= {f"Hybrid_margin_{level}": dict(level=level, margin=0.05, default="Ens_top3") for level in levels}

picks = {}
hybrid_runs = []
for d in hybrid_decisions:
    evidence["hybrid"] = common_rows(pd.concat([evidence["stats+ml"], ensemble_evidence(d)]))
    hybrid_runs += [strategy(name, "hybrid", [d], picks=picks, **policy) for name, policy in hybrid_policies.items()]
hybrid_runs = pd.concat(hybrid_runs)

### Comparison on identical rows

Against the standalone ensemble, the best individual model chosen on development evidence (`Best_P5_all_hist`), and the P4 and P5 tournaments from the previous section. All strategies are scored together by `leaderboard` on the decisions where the hybrid exists. The final window is exploratory.

In [31]:
standalone = pd.concat([dev_strategies, strategies])
standalone = standalone[standalone.model == "Average_top3_ml"]
rivals = list(tour.model[tour.pool.isin(["P4_hist_features", "P5_all_hist"])]) + ["Best_P5_all_hist"]
hybrid_all = pd.concat([hybrid_runs, standalone, pool_runs[pool_runs.model.isin(rivals)]])
hybrid_all = hybrid_all[hybrid_all.cutoff.isin(hybrid_decisions)]

families = {"P4 tournament": tour[tour.pool == "P4_hist_features"], "P5 tournament": tour[tour.pool == "P5_all_hist"]}
design = pd.concat([f.assign(family=name) for name, f in families.items()]
                   + [pd.DataFrame({"model": [f"{p}_{level}" for level in levels], "level": levels, "family": name}) for p, name in [("Hybrid", "Hybrid"), ("Hybrid_margin", "Hybrid margin")]])


def by_family(values):
    return design.assign(value=design.model.map(values)).pivot(index="family", columns="level", values="value").loc[["P4 tournament", "P5 tournament", "Hybrid", "Hybrid margin"], levels]


hybrid_board = leaderboard(hybrid_all, actuals, scope="test").set_index("model")
display(hybrid_board.loc[["Average_top3_ml", "Best_P5_all_hist"], ["wape", "bias", "mean_rmsse", "apc", "coverage"]].round(3))
pd.concat({metric: by_family(hybrid_board[metric]) for metric in ["wape", "bias", "mean_rmsse", "apc", "coverage"]}, axis=1).round(3)

,wape,bias,mean_rmsse,apc,coverage
model,,,,,
Average_top3_ml,0.364,-0.050,1.266,0.104,1.0
Best_P5_all_hist,0.369,-0.003,1.265,0.122,1.0


wape                                        bias  \
level         portfolio department  class subclass    sku portfolio   
family                                                                
P4 tournament     0.367      0.406  0.406    0.420  0.409    -0.101   
P5 tournament     0.369      0.376  0.370    0.385  0.395    -0.003   
Hybrid            0.369      0.375  0.370    0.385  0.395    -0.003   
Hybrid margin     0.364      0.377  0.367    0.381  0.392    -0.050   

                                                 ...       apc             \
level         department  class subclass    sku  ... portfolio department   
family                                           ...                        
P4 tournament      0.029  0.065    0.089  0.067  ...     0.140      0.097   
P5 tournament     -0.004 -0.004    0.025  0.039  ...     0.122      0.137   
Hybrid            -0.004 -0.005    0.024  0.038  ...     0.122      0.135   
Hybrid margin     -0.008 -0.008    0.024  0.040  ...     0.104      0.132   

                                      coverage                                 
level          class subclass    sku portfolio department class subclass  sku  
family                                                                         
P4 tournament  0.108    0.116  0.114       1.0        1.0   1.0      1.0  1.0  
P5 tournament  0.120    0.124  0.120       1.0        1.0   1.0      1.0  1.0  
Hybrid         0.118    0.124  0.120       1.0        1.0   1.0      1.0  1.0  
Hybrid margin  0.117    0.123  0.121       1.0        1.0   1.0      1.0  1.0  

[4 rows x 25 columns]

### Share of SKUs that keep the ensemble

Percentage of SKUs whose chosen model is `Ens_top3`, at each decision. The rest switch to another candidate. For Hybrid_margin the groups that stay did not clear the 5% margin.

In [32]:
kept = pd.Series({key: (choice == "Ens_top3").mean() * 100 for key, choice in picks.items()})
kept.index = pd.MultiIndex.from_tuples(kept.index, names=["strategy", "cutoff"])
kept.unstack("cutoff").loc[list(hybrid_policies)].round(0)

cutoff,2024-12-04,2025-01-01,2025-02-12,2025-03-12,2025-04-09
strategy,,,,,
Hybrid_portfolio,0.0,0.0,0.0,0.0,0.0
Hybrid_department,1.0,9.0,8.0,10.0,10.0
Hybrid_class,2.0,8.0,5.0,7.0,8.0
Hybrid_subclass,2.0,4.0,2.0,2.0,2.0
Hybrid_sku,3.0,2.0,1.0,1.0,1.0
Hybrid_margin_portfolio,0.0,100.0,100.0,100.0,100.0
Hybrid_margin_department,16.0,51.0,52.0,58.0,55.0
Hybrid_margin_class,19.0,38.0,38.0,37.0,42.0
Hybrid_margin_subclass,15.0,30.0,31.0,31.0,32.0


### Consistency across decision dates

WAPE at each of the five decisions: two development decisions (two and three evidence runs, 2024-12-04 and 2025-01-01) and three later cutoffs (four to seven runs). Counts are out of five. Changes are in WAPE points against the standalone ensemble; negative is better.

In [33]:
by_cutoff = pd.concat([score(hybrid_all[hybrid_all.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in hybrid_decisions])
wape_c = by_cutoff.pivot(index="model", columns="cutoff", values="wape")
vs_ensemble = wape_c.loc[design.model] - wape_c.loc["Average_top3_ml"]
vs_p5 = wape_c.loc[design.model].values - wape_c.loc[design.level.map(tour[tour.pool == "P5_all_hist"].set_index("level").model)].values
vs_p5 = pd.DataFrame(vs_p5, index=design.model, columns=wape_c.columns)
early, late = hybrid_decisions[:2], hybrid_decisions[2:]

pd.concat({
    "beats ensemble": by_family((vs_ensemble < 0).sum(axis=1)),
    "beats P5 tournament": by_family((vs_p5 < 0).sum(axis=1)),
    "change vs ensemble, development": by_family(vs_ensemble[early].mean(axis=1) * 100).round(2),
    "change vs ensemble, later cutoffs": by_family(vs_ensemble[late].mean(axis=1) * 100).round(2),
}, axis=1)

beats ensemble                                \
level              portfolio department class subclass sku   
family                                                       
P4 tournament              0          0     0        1   1   
P5 tournament              3          1     1        3   2   
Hybrid                     3          1     1        3   2   
Hybrid margin              0          1     3        3   3   

              beats P5 tournament                                \
level                   portfolio department class subclass sku   
family                                                            
P4 tournament                   2          0     0        0   2   
P5 tournament                   0          0     0        0   0   
Hybrid                          0          5     4        3   4   
Hybrid margin                   1          1     3        5   5   

              change vs ensemble, development                                  \
level                               portfolio department class subclass   sku   
family                                                                          
P4 tournament                            0.55       2.25  1.17    -0.38 -1.45   
P5 tournament                            0.51       1.32 -0.09    -1.56 -1.57   
Hybrid                                   0.51       1.30 -0.14    -1.58 -1.58   
Hybrid margin                            0.63       1.35 -0.19    -1.94 -2.06   

              change vs ensemble, later cutoffs                            \
level                                 portfolio department class subclass   
family                                                                      
P4 tournament                              0.82       2.42  2.31     2.40   
P5 tournament                             -0.14       0.61  0.32     0.74   
Hybrid                                    -0.14       0.55  0.29     0.75   
Hybrid margin                              0.00       0.74  0.19     0.50   

                     
level           sku  
family               
P4 tournament  2.16  
P5 tournament  1.69  
Hybrid         1.68  
Hybrid margin  1.36

### Reading the ensemble results

Final window, exploratory.

- **Adding the ensemble to the pool changes nothing.** Hybrid is within 0.001 WAPE of the P5 tournament at every level. The tournament rarely picks the ensemble: 10% of SKUs at department level, 8% at class, 2% at subclass and 1% at SKU.
- **The 5% margin policy keeps the ensemble where evidence is thin and moves SKUs elsewhere as groups get smaller.** At the final decision 55% of SKUs keep the ensemble at department level, 42% at class, 32% at subclass and 24% at SKU. At portfolio level it always keeps it, so that strategy is the standalone ensemble (0.364).
- **Margin selection beats the plain tournament at fine levels.** It is 0.003 to 0.004 WAPE better than the P5 tournament at class, subclass and SKU, and better on all five dates at subclass and SKU. At department level it is 0.001 worse.
- **No selection policy beats the standalone ensemble in the final window.** Hybrid_margin is 0.367 at class, 0.377 at department, 0.381 at subclass and 0.392 at SKU, against 0.364. The best individual model chosen on development evidence is 0.369.
- **Bias.** The ensemble is -0.050. Margin selection moves it to -0.008 at class and +0.024 and +0.040 at subclass and SKU. Mean RMSSE is about 1.27 for the ensemble and the best individual.

Consistency across decision dates:

- At the two development decisions (two and three evidence runs), Hybrid_margin at subclass and SKU beats the ensemble by about 1.9 and 2.1 WAPE points on average, and class by 0.2.
- At the three later cutoffs every margin policy except portfolio is worse than the ensemble, by 0.2 (class) to 1.4 points (SKU) on average. Over the five dates, class, subclass and SKU beat the ensemble on three.
- The ensemble is a hard baseline: selection gains over it appear when evidence is thin and disappear as evidence grows. Five overlapping windows are weak evidence, so treat differences under a point as noise.
- The ensemble's evidence uses leave-one-out membership, so it may be slightly pessimistic about the ensemble compared with its decision-date membership.

## Optional: adding CatBoost candidates

The three CatBoost models from notebook 03 join the 18 historical-only candidates (`P6_catboost` = P5 plus CatBoost). `lgbm_exog_categorical` stays out, so the only change to the pool is CatBoost. The existing policies are unchanged; they are rerun on the larger pool and compared with the P5 versions on identical rows:

- the best individual model on completed development runs
- the top-three ML average, members chosen on completed development runs from the ML models plus CatBoost
- portfolio, department, class, subclass and SKU tournaments with pooled MAE

In [34]:
boosting = load_forecasts(["03_catboost"])
catboost = sorted(m for m in boosting.model.unique() if m.startswith("catboost"))
boosting = boosting[boosting.model.isin(catboost)]

candidates = pd.concat([candidates, boosting])
tier_of = pd.concat([tier_of, boosting.drop_duplicates("model").set_index("model").information_tier])
nested["P6_catboost"] = nested["P5_all_hist"] + catboost
pools["P6_catboost"] = nested["P6_catboost"]
catboost_dev = boosting[boosting.cutoff.isin(DEV_CUTOFFS)].merge(actuals, on=["unique_id", "ds"])
evidence["P6_catboost"] = common_rows(pd.concat([evidence["stats+ml"], catboost_dev]))


def top3_among(models, runs):
    rows = candidates[candidates.model.isin(models) & candidates.cutoff.isin(runs)]
    return score(rows, actuals, rmsse=False).sort_values("wape").model.head(3).tolist()


catboost_picks = {}
catboost_runs = [strategy(f"Tour_P6_catboost_{level}", "P6_catboost", all_decisions, picks=catboost_picks, level=level) for level in levels]
catboost_runs += [average("Average_top3_ml_catboost", top3_among(ml + catboost, completed_runs(d)), [d]) for d in all_decisions]
catboost_best = {d: best_individual("P6_catboost", d) for d in all_decisions}
catboost_runs = pd.concat(catboost_runs + [rows for rows, name in catboost_best.values()])

Which models the top-three average and the best-individual rule pick at the final decision, with and without CatBoost:

In [35]:
pd.DataFrame({
    "best individual": [best_individual("P5_all_hist", TEST_CUTOFF)[1], catboost_best[TEST_CUTOFF][1]],
    "top-three members": [", ".join(top3_of(completed_runs(TEST_CUTOFF))), ", ".join(top3_among(ml + catboost, completed_runs(TEST_CUTOFF)))],
}, index=["without CatBoost", "with CatBoost"])

,best individual,top-three members
without CatBoost,lgbm_l1_std,"lgbm_l1_std, lgbm_promo_calendar_none, lgbm_pr..."
with CatBoost,lgbm_l1_std,"lgbm_l1_std, lgbm_promo_calendar_none, lgbm_pr..."


### With and without CatBoost, on identical rows

Final 14 weeks (exploratory) from `leaderboard`, then WAPE at each of the six decision dates. Changes are in WAPE points; negative means adding CatBoost helped.

In [36]:
p5 = tour[tour.pool == "P5_all_hist"].set_index("level").model
pairs = pd.DataFrame({
    "without": ["Best_P5_all_hist", "Average_top3_ml"] + [p5[level] for level in levels],
    "with": ["Best_P6_catboost", "Average_top3_ml_catboost"] + [f"Tour_P6_catboost_{level}" for level in levels],
}, index=["best individual", "top-three average"] + [f"tournament: {level}" for level in levels])

compare = pd.concat([pool_runs[pool_runs.model.isin(pairs.without)], standalone, catboost_runs]).drop_duplicates(["unique_id", "cutoff", "ds", "model"])
catboost_board = leaderboard(compare, actuals, scope="test").set_index("model")
metrics = ["wape", "bias", "mean_rmsse", "apc", "coverage"]
pd.concat({"without": catboost_board.loc[pairs.without, metrics].set_axis(pairs.index),
           "with": catboost_board.loc[pairs["with"], metrics].set_axis(pairs.index)}, axis=1).round(3)

without                                     with  \
                          wape   bias mean_rmsse    apc coverage   wape   
best individual          0.369 -0.003      1.265  0.122      1.0  0.369   
top-three average        0.364 -0.050      1.266  0.104      1.0  0.364   
tournament: portfolio    0.369 -0.003      1.265  0.122      1.0  0.369   
tournament: department   0.376 -0.004      1.279  0.138      1.0  0.377   
tournament: class        0.370 -0.004      1.295  0.122      1.0  0.371   
tournament: subclass     0.385  0.025      1.447  0.127      1.0  0.385   
tournament: sku          0.395  0.039      1.316  0.121      1.0  0.395   

                                                          
                         bias mean_rmsse    apc coverage  
best individual        -0.003      1.265  0.122      1.0  
top-three average      -0.050      1.266  0.104      1.0  
tournament: portfolio  -0.003      1.265  0.122      1.0  
tournament: department -0.003      1.279  0.137      1.0  
tournament: class      -0.004      1.295  0.121      1.0  
tournament: subclass    0.025      1.448  0.127      1.0  
tournament: sku         0.039      1.316  0.120      1.0

In [37]:
by_cutoff = pd.concat([score(compare[compare.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in all_decisions])
catboost_wape = by_cutoff.pivot(index="model", columns="cutoff", values="wape")
change = (catboost_wape.loc[pairs["with"]].values - catboost_wape.loc[pairs.without].values) * 100
change = pd.DataFrame(change, index=pairs.index, columns=catboost_wape.columns).round(2)
change.assign(helped=(change < 0).sum(axis=1))

cutoff,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-02-12 00:00:00,2025-03-12 00:00:00,2025-04-09 00:00:00,helped
best individual,0.0,0.00,0.00,0.00,0.00,0.00,0
top-three average,0.0,0.00,0.00,0.00,0.00,0.00,0
tournament: portfolio,0.0,0.00,0.00,0.00,0.00,0.00,0
tournament: department,0.0,-0.00,0.01,0.00,-0.01,0.03,1
tournament: class,0.0,0.03,0.04,-0.01,-0.00,0.04,1
tournament: subclass,0.0,-0.08,-0.01,-0.05,-0.01,0.04,4
tournament: sku,0.0,-0.08,-0.09,-0.10,-0.03,0.03,4


### How often CatBoost is chosen

Share of SKUs whose tournament winner is a CatBoost model, at each decision.

In [38]:
chosen = pd.Series({key: choice.isin(catboost).mean() * 100 for key, choice in catboost_picks.items()})
chosen.index = pd.MultiIndex.from_tuples(chosen.index, names=["strategy", "cutoff"])
chosen.unstack("cutoff").loc[[f"Tour_P6_catboost_{level}" for level in levels]].round(0)

cutoff,2024-11-06,2024-12-04,2025-01-01,2025-02-12,2025-03-12,2025-04-09
strategy,,,,,,
Tour_P6_catboost_portfolio,0.0,0.0,0.0,0.0,0.0,0.0
Tour_P6_catboost_department,2.0,1.0,1.0,2.0,1.0,1.0
Tour_P6_catboost_class,1.0,3.0,2.0,2.0,3.0,2.0
Tour_P6_catboost_subclass,2.0,2.0,3.0,3.0,4.0,4.0
Tour_P6_catboost_sku,2.0,3.0,4.0,5.0,5.0,5.0


### Reading the CatBoost results

- **Best individual and top-three average: no change.** CatBoost is never the best candidate on completed development runs and never among the top three, so both strategies pick the same models as before and score the same (0.369 and 0.364).
- **Tournaments: no meaningful change.** At every level and decision date the change is within 0.1 WAPE points. Subclass and SKU are marginally better at four of six dates and marginally worse in the final window. Tournaments choose a CatBoost model for 1% to 5% of SKUs.
- **Diversity.** With these selection rules CatBoost adds no useful diversity: where it is chosen, it replaces a LightGBM forecast that is about as good.

`lgbm_exog_categorical` is not in this pool. It improved the most in notebook 03, and adding it is the more promising next test.

## Optional: adding the categorical LightGBM candidates

`lgbm_l1_std_categorical` and `lgbm_promo_calendar_categorical` from notebook 03 join the 18 historical-only candidates (`P7_categorical` = P5 plus the two). CatBoost is not included. The same policies as in the CatBoost section run on this pool and are compared with the P5 versions on identical rows.

In [39]:
categorical = ["lgbm_l1_std_categorical", "lgbm_promo_calendar_categorical"]
cat_forecasts = load_forecasts(["03_categorical"])

candidates = pd.concat([candidates, cat_forecasts])
tier_of = pd.concat([tier_of, cat_forecasts.drop_duplicates("model").set_index("model").information_tier])
nested["P7_categorical"] = nested["P5_all_hist"] + categorical
pools["P7_categorical"] = nested["P7_categorical"]
cat_dev = cat_forecasts[cat_forecasts.cutoff.isin(DEV_CUTOFFS)].merge(actuals, on=["unique_id", "ds"])
evidence["P7_categorical"] = common_rows(pd.concat([evidence["stats+ml"], cat_dev]))

categorical_picks = {}
categorical_runs = [strategy(f"Tour_P7_categorical_{level}", "P7_categorical", all_decisions, picks=categorical_picks, level=level) for level in levels]
categorical_runs += [average("Average_top3_ml_categorical", top3_among(ml + categorical, completed_runs(d)), [d]) for d in all_decisions]
categorical_best = {d: best_individual("P7_categorical", d) for d in all_decisions}
categorical_runs = pd.concat(categorical_runs + [rows for rows, name in categorical_best.values()])

In [40]:
pd.DataFrame({
    "best individual": [best_individual("P5_all_hist", TEST_CUTOFF)[1], categorical_best[TEST_CUTOFF][1]],
    "top-three members": [", ".join(top3_of(completed_runs(TEST_CUTOFF))), ", ".join(top3_among(ml + categorical, completed_runs(TEST_CUTOFF)))],
}, index=["P5", "P7_categorical"])

,best individual,top-three members
P5,lgbm_l1_std,"lgbm_l1_std, lgbm_promo_calendar_none, lgbm_pr..."
P7_categorical,lgbm_l1_std_categorical,"lgbm_l1_std_categorical, lgbm_l1_std, lgbm_pro..."


### With and without the categorical candidates, on identical rows

In [41]:
pairs = pd.DataFrame({
    "without": ["Best_P5_all_hist", "Average_top3_ml"] + [p5[level] for level in levels],
    "with": ["Best_P7_categorical", "Average_top3_ml_categorical"] + [f"Tour_P7_categorical_{level}" for level in levels],
}, index=["best individual", "top-three average"] + [f"tournament: {level}" for level in levels])

compare = pd.concat([pool_runs[pool_runs.model.isin(pairs.without)], standalone, categorical_runs]).drop_duplicates(["unique_id", "cutoff", "ds", "model"])
categorical_board = leaderboard(compare, actuals, scope="test").set_index("model")
pd.concat({"without": categorical_board.loc[pairs.without, metrics].set_axis(pairs.index),
           "with": categorical_board.loc[pairs["with"], metrics].set_axis(pairs.index)}, axis=1).round(3)

without                                     with  \
                          wape   bias mean_rmsse    apc coverage   wape   
best individual          0.369 -0.003      1.265  0.121      1.0  0.350   
top-three average        0.364 -0.050      1.266  0.104      1.0  0.348   
tournament: portfolio    0.369 -0.003      1.265  0.121      1.0  0.350   
tournament: department   0.376 -0.004      1.279  0.138      1.0  0.369   
tournament: class        0.370 -0.004      1.295  0.122      1.0  0.368   
tournament: subclass     0.385  0.025      1.447  0.127      1.0  0.378   
tournament: sku          0.395  0.039      1.316  0.121      1.0  0.387   

                                                          
                         bias mean_rmsse    apc coverage  
best individual        -0.020      1.257  0.116      1.0  
top-three average      -0.038      1.243  0.105      1.0  
tournament: portfolio  -0.020      1.257  0.116      1.0  
tournament: department -0.001      1.275  0.114      1.0  
tournament: class      -0.007      1.294  0.124      1.0  
tournament: subclass    0.012      1.445  0.138      1.0  
tournament: sku         0.028      1.311  0.125      1.0

In [42]:
by_cutoff = pd.concat([score(compare[compare.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in all_decisions])
categorical_wape = by_cutoff.pivot(index="model", columns="cutoff", values="wape")
change = (categorical_wape.loc[pairs["with"]].values - categorical_wape.loc[pairs.without].values) * 100
change = pd.DataFrame(change, index=pairs.index, columns=categorical_wape.columns).round(2)
change.assign(helped=(change < 0).sum(axis=1))

cutoff,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-02-12 00:00:00,2025-03-12 00:00:00,2025-04-09 00:00:00,helped
best individual,-1.64,-4.76,-2.17,-1.99,-1.08,-1.84,6
top-three average,-1.67,-1.24,-1.72,-1.65,-1.90,-1.64,6
tournament: portfolio,-1.64,-4.76,-2.17,-1.99,-1.08,-1.84,6
tournament: department,0.12,-1.04,-0.99,-0.76,-1.41,-0.76,5
tournament: class,-0.12,-0.55,0.06,-0.52,-0.03,-0.27,5
tournament: subclass,-0.20,-0.21,-0.13,-0.75,-0.33,-0.72,6
tournament: sku,-0.22,0.17,0.06,-0.66,-0.46,-0.82,4


Share of SKUs whose tournament winner is one of the two categorical models, at each decision.

In [43]:
chosen = pd.Series({key: choice.isin(categorical).mean() * 100 for key, choice in categorical_picks.items()})
chosen.index = pd.MultiIndex.from_tuples(chosen.index, names=["strategy", "cutoff"])
chosen.unstack("cutoff").loc[[f"Tour_P7_categorical_{level}" for level in levels]].round(0)

cutoff,2024-11-06,2024-12-04,2025-01-01,2025-02-12,2025-03-12,2025-04-09
strategy,,,,,,
Tour_P7_categorical_portfolio,100.0,100.0,100.0,100.0,100.0,100.0
Tour_P7_categorical_department,16.0,17.0,22.0,18.0,40.0,30.0
Tour_P7_categorical_class,21.0,23.0,22.0,21.0,26.0,27.0
Tour_P7_categorical_subclass,14.0,17.0,17.0,15.0,20.0,19.0
Tour_P7_categorical_sku,11.0,12.0,13.0,13.0,13.0,14.0


### Reading the categorical results

- **Best individual.** On completed development runs `lgbm_l1_std_categorical` replaces `lgbm_l1_std` as the best candidate: 0.350 against 0.369 in the final window, better at all six decision dates.
- **Top-three average.** The members become `lgbm_l1_std_categorical`, `lgbm_l1_std` and `lgbm_promo_calendar_categorical`. The average scores 0.348 against 0.364, better at all six dates, and is the best historical-only strategy in Day 1. Its bias is -0.038 against -0.050.
- **Tournaments.** Every level improves in the final window: portfolio by 1.8 points (it picks the same model as the best individual), department 0.8, class 0.3, subclass 0.7, SKU 0.8. Department, class, subclass and SKU improve at four to six of six dates. Below portfolio level the two new models win 11% to 40% of SKUs.
- **Ranking of policies is unchanged.** The finer levels are still worse than portfolio selection and the top-three average, so the gain comes from a stronger candidate, not from more granular selection.

Unlike CatBoost, these candidates add measurable value: they raise the strongest single model, and the top-three average and every tournament inherit the gain.

## Optional: adding the historical-behavior candidates

All eleven models from notebook 03c join the `P7_categorical` pool (`P8_behavior`). Which of them matter is left to the selection rules and completed development evidence, as for every other candidate. The same policies run on P8 and are compared with P7 on identical rows.

In [44]:
behavior_forecasts = load_forecasts(["03c_behavior_features"])
behavior = sorted(behavior_forecasts.model.unique())

candidates = pd.concat([candidates, behavior_forecasts])
tier_of = pd.concat([tier_of, behavior_forecasts.drop_duplicates("model").set_index("model").information_tier])
nested["P8_behavior"] = nested["P7_categorical"] + behavior
pools["P8_behavior"] = nested["P8_behavior"]
behavior_dev = behavior_forecasts[behavior_forecasts.cutoff.isin(DEV_CUTOFFS)].merge(actuals, on=["unique_id", "ds"])
evidence["P8_behavior"] = common_rows(pd.concat([evidence["P7_categorical"], behavior_dev]))

behavior_picks = {}
behavior_runs = [strategy(f"Tour_P8_behavior_{level}", "P8_behavior", all_decisions, picks=behavior_picks, level=level) for level in levels]
behavior_runs += [average("Average_top3_ml_behavior", top3_among(ml + categorical + behavior, completed_runs(d)), [d]) for d in all_decisions]
behavior_best = {d: best_individual("P8_behavior", d) for d in all_decisions}
behavior_runs = pd.concat(behavior_runs + [rows for rows, name in behavior_best.values()])

In [45]:
pd.DataFrame({
    "best individual": [categorical_best[TEST_CUTOFF][1], behavior_best[TEST_CUTOFF][1]],
    "top-three members": [top3_among(ml + categorical, completed_runs(TEST_CUTOFF)), top3_among(ml + categorical + behavior, completed_runs(TEST_CUTOFF))],
}, index=["P7_categorical", "P8_behavior"])

,best individual,top-three members
P7_categorical,lgbm_l1_std_categorical,"[lgbm_l1_std_categorical, lgbm_l1_std, lgbm_pr..."
P8_behavior,lgbm_l1_cat_price_response,"[lgbm_l1_cat_price_response, lgbm_l1_cat_combi..."


### With and without the behavior candidates, on identical rows

In [46]:
pairs = pd.DataFrame({
    "without": ["Best_P7_categorical", "Average_top3_ml_categorical"] + [f"Tour_P7_categorical_{level}" for level in levels],
    "with": ["Best_P8_behavior", "Average_top3_ml_behavior"] + [f"Tour_P8_behavior_{level}" for level in levels],
}, index=["best individual", "top-three average"] + [f"tournament: {level}" for level in levels])

compare = pd.concat([categorical_runs, behavior_runs]).drop_duplicates(["unique_id", "cutoff", "ds", "model"])
behavior_board = leaderboard(compare, actuals, scope="test").set_index("model")
pd.concat({"without": behavior_board.loc[pairs.without, metrics].set_axis(pairs.index),
           "with": behavior_board.loc[pairs["with"], metrics].set_axis(pairs.index)}, axis=1).round(3)

without                                     with  \
                          wape   bias mean_rmsse    apc coverage   wape   
best individual          0.350 -0.020      1.257  0.116      1.0  0.344   
top-three average        0.348 -0.038      1.243  0.105      1.0  0.345   
tournament: portfolio    0.350 -0.020      1.257  0.116      1.0  0.344   
tournament: department   0.369 -0.001      1.275  0.114      1.0  0.369   
tournament: class        0.368 -0.007      1.294  0.124      1.0  0.371   
tournament: subclass     0.378  0.012      1.445  0.138      1.0  0.370   
tournament: sku          0.387  0.028      1.311  0.126      1.0  0.380   

                                                          
                         bias mean_rmsse    apc coverage  
best individual        -0.035      1.249  0.110      1.0  
top-three average      -0.032      1.249  0.107      1.0  
tournament: portfolio  -0.035      1.249  0.110      1.0  
tournament: department  0.002      1.266  0.107      1.0  
tournament: class      -0.005      1.294  0.103      1.0  
tournament: subclass   -0.007      1.303  0.134      1.0  
tournament: sku         0.014      1.306  0.128      1.0

In [47]:
by_cutoff = pd.concat([score(compare[compare.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in all_decisions])
behavior_wape = by_cutoff.pivot(index="model", columns="cutoff", values="wape")
change = (behavior_wape.loc[pairs["with"]].values - behavior_wape.loc[pairs.without].values) * 100
change = pd.DataFrame(change, index=pairs.index, columns=behavior_wape.columns).round(2)
change.assign(helped=(change < 0).sum(axis=1))

cutoff,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-02-12 00:00:00,2025-03-12 00:00:00,2025-04-09 00:00:00,helped
best individual,-0.20,-0.39,-0.12,0.02,-0.46,-0.66,5
top-three average,-1.34,-2.67,-1.02,-1.35,0.21,-0.32,5
tournament: portfolio,-0.20,-0.39,-0.12,0.02,-0.46,-0.66,5
tournament: department,-0.64,-2.97,-2.45,-0.33,-0.23,0.00,5
tournament: class,-0.80,-0.79,-1.02,-1.21,-0.57,0.37,5
tournament: subclass,-0.69,-0.55,-0.96,-0.64,-0.63,-0.78,6
tournament: sku,-0.43,-0.65,-1.06,-0.75,-0.43,-0.67,6


Share of SKUs whose tournament winner is one of the behavior models, at each decision.

In [48]:
chosen = pd.Series({key: choice.isin(behavior).mean() * 100 for key, choice in behavior_picks.items()})
chosen.index = pd.MultiIndex.from_tuples(chosen.index, names=["strategy", "cutoff"])
chosen.unstack("cutoff").loc[[f"Tour_P8_behavior_{level}" for level in levels]].round(0)

cutoff,2024-11-06,2024-12-04,2025-01-01,2025-02-12,2025-03-12,2025-04-09
strategy,,,,,,
Tour_P8_behavior_portfolio,100.0,100.0,100.0,100.0,100.0,100.0
Tour_P8_behavior_department,32.0,45.0,40.0,30.0,55.0,58.0
Tour_P8_behavior_class,31.0,34.0,36.0,38.0,42.0,39.0
Tour_P8_behavior_subclass,24.0,29.0,29.0,26.0,31.0,31.0
Tour_P8_behavior_sku,18.0,20.0,19.0,19.0,19.0,20.0


### Reading the behavior results

- **Best individual.** `lgbm_l1_cat_price_response` replaces `lgbm_l1_std_categorical`: 0.344 against 0.350 in the final window, better at five of six decision dates.
- **Top-three average.** 0.345 against 0.348, better at five of six dates. Its members are now `lgbm_l1_cat_price_response`, `lgbm_l1_cat_combined` and `lgbm_l1_cat_price`, three versions of nearly the same feature set. Choosing members by development WAPE alone picks near-duplicates, so the average adds almost nothing over its best member, unlike the P7 average.
- **Tournaments.** Subclass (0.378 to 0.370) and SKU (0.387 to 0.380) improve at all six dates. Department is unchanged in the final window and class is slightly worse, though both improve at five of six dates. Below portfolio level the behavior models win 19% to 58% of SKUs.
- **Ranking of policies is unchanged.** Portfolio selection and the top-three average still beat every finer level.

The gain to the system is about 0.3 to 0.8 points where it appears, carried by the price-history features. It shows up in the single best model and the system inherits it.

## Optional: ensemble diversity

The P8 top-three average contains three near-duplicate LightGBM variants and adds almost nothing over its best member. Would members with complementary errors make a better ensemble? No model is retrained; every ensemble is an equal-weight average of saved historical-only forecasts.

The pool is every historical-only individual model in Day 1: P8 plus the CatBoost models and `lgbm_exog_categorical`. There is no AutoETS in Day 1. Each model is assigned a family:

- **statistical**: Naive, HistoricAverage, SeasonalNaive
- **lgbm_l1**: LightGBM with the L1 objective, including the categorical and behavior variants
- **lgbm_other**: other LightGBM objectives, horizon strategies and pooling
- **catboost**

In [49]:
pd.set_option("display.max_colwidth", None)
candidates = pd.concat([candidates, load_forecasts(["03_catboost"]).query("model == 'lgbm_exog_categorical'")])
tier_of["lgbm_exog_categorical"] = "historical_only"
pool = nested["P8_behavior"] + catboost + ["lgbm_exog_categorical"]
family = {m: "statistical" if m in stats else "catboost" if m.startswith("catboost") else "lgbm_l1" if m.startswith("lgbm_l1") else "lgbm_other" for m in pool}


def residuals(runs):
    """Forecast errors (forecast - actual) of every pool model, one column per model, on rows every model forecasts."""
    rows = common_rows(candidates[candidates.model.isin(pool) & candidates.cutoff.isin(runs)]).merge(actuals, on=["unique_id", "ds"])
    errors = rows.assign(error=rows.yhat - rows.y).pivot(index=["unique_id", "cutoff", "ds"], columns="model", values="error")
    y = rows.drop_duplicates(["unique_id", "cutoff", "ds"]).set_index(["unique_id", "cutoff", "ds"]).y.reindex(errors.index)
    return errors, y


errors, y = residuals(completed_runs(TEST_CUTOFF))
single = errors.abs().sum() / y.sum()
pd.DataFrame({"family": family, "dev_wape": single}).sort_values("dev_wape").groupby("family").head(3).round(3)

,family,dev_wape
lgbm_l1_cat_price_response,lgbm_l1,0.353
lgbm_l1_cat_combined,lgbm_l1,0.353
lgbm_l1_cat_price,lgbm_l1,0.354
lgbm_promo_calendar_categorical,lgbm_other,0.394
lgbm_promo_calendar_none,lgbm_other,0.396
lgbm_promo_none,lgbm_other,0.396
SeasonalNaive,statistical,0.408
HistoricAverage,statistical,0.426
catboost_exog_categorical,catboost,0.434
catboost_hist_exog,catboost,0.438


### How different are the errors?

On the seven development runs, for the best two models of each family: the correlation of forecast errors, and the share of product-weeks where two models err in opposite directions. Errors are in units, so the correlation is weighted toward high-volume products, like WAPE.

In [50]:
shortlist = single.groupby(family).nsmallest(2).index.get_level_values(-1)
correlation = errors[shortlist].corr()
opposite = pd.DataFrame({a: {b: (np.sign(errors[a]) != np.sign(errors[b])).mean() for b in shortlist} for a in shortlist})
pd.concat({"error correlation": correlation, "opposite sign": opposite}, axis=1).round(2)

error correlation                     \
model                           catboost_exog_categorical catboost_hist_exog   
catboost_exog_categorical                            1.00               0.99   
catboost_hist_exog                                   0.99               1.00   
lgbm_l1_cat_price_response                           0.82               0.82   
lgbm_l1_cat_combined                                 0.83               0.82   
lgbm_promo_calendar_categorical                      0.85               0.84   
lgbm_promo_calendar_none                             0.91               0.90   
SeasonalNaive                                        0.62               0.61   
HistoricAverage                                      0.89               0.89   

                                                            \
model                           lgbm_l1_cat_price_response   
catboost_exog_categorical                             0.82   
catboost_hist_exog                                    0.82   
lgbm_l1_cat_price_response                            1.00   
lgbm_l1_cat_combined                                  1.00   
lgbm_promo_calendar_categorical                       0.92   
lgbm_promo_calendar_none                              0.91   
SeasonalNaive                                         0.60   
HistoricAverage                                       0.89   

                                                      \
model                           lgbm_l1_cat_combined   
catboost_exog_categorical                       0.83   
catboost_hist_exog                              0.82   
lgbm_l1_cat_price_response                      1.00   
lgbm_l1_cat_combined                            1.00   
lgbm_promo_calendar_categorical                 0.91   
lgbm_promo_calendar_none                        0.91   
SeasonalNaive                                   0.60   
HistoricAverage                                 0.89   

                                                                 \
model                           lgbm_promo_calendar_categorical   
catboost_exog_categorical                                  0.85   
catboost_hist_exog                                         0.84   
lgbm_l1_cat_price_response                                 0.92   
lgbm_l1_cat_combined                                       0.91   
lgbm_promo_calendar_categorical                            1.00   
lgbm_promo_calendar_none                                   0.96   
SeasonalNaive                                              0.60   
HistoricAverage                                            0.91   

                                                                        \
model                           lgbm_promo_calendar_none SeasonalNaive   
catboost_exog_categorical                           0.91          0.62   
catboost_hist_exog                                  0.90          0.61   
lgbm_l1_cat_price_response                          0.91          0.60   
lgbm_l1_cat_combined                                0.91          0.60   
lgbm_promo_calendar_categorical                     0.96          0.60   
lgbm_promo_calendar_none                            1.00          0.62   
SeasonalNaive                                       0.62          1.00   
HistoricAverage                                     0.93          0.68   

                                                            opposite sign  \
model                           HistoricAverage catboost_exog_categorical   
catboost_exog_categorical                  0.89                      0.00   
catboost_hist_exog                         0.89                      0.02   
lgbm_l1_cat_price_response                 0.89                      0.17   
lgbm_l1_cat_combined                       0.89                      0.17   
lgbm_promo_calendar_categorical            0.91                      0.14   
lgbm_promo_calendar_none                   0.93                      0.11   
SeasonalNaive                

The near-duplicates in the P8 top three:

In [51]:
p8_top3 = top3_among(ml + categorical + behavior, completed_runs(TEST_CUTOFF))
errors[p8_top3].corr().round(3)

model,lgbm_l1_cat_price_response,lgbm_l1_cat_combined,lgbm_l1_cat_price
model,,,
lgbm_l1_cat_price_response,1.000,0.997,0.995
lgbm_l1_cat_combined,0.997,1.000,0.995
lgbm_l1_cat_price,0.995,0.995,1.000


### Ensemble rules

Each rule takes the completed runs available on a decision date and returns members. All use equal weights.

- **C**: the best model and the best statistical model.
- **D**: the best model and the best model from another machine-learning family.
- **E**: the best model, then twice the competitive model (development WAPE within 3 points of the best) least correlated with the members so far.
- **F**: the best model and the best model of each of the two next-best families.

A is the P8 top-three average and B the best individual model, as above.

In [52]:
def ensemble_rules(runs):
    errors, y = residuals(runs)
    single = errors.abs().sum() / y.sum()
    best = single.idxmin()
    best_of = lambda fams: single[[m for m in single.index if family[m] in fams]].idxmin()
    correlation = errors.corr()

    competitive = single[single <= single.min() + 0.03].index
    diverse = [best]
    while len(diverse) < 3:
        diverse.append(correlation.loc[competitive.difference(diverse), diverse].mean(axis=1).idxmin())

    other_families = single.groupby(family).min().drop(family[best]).nsmallest(2).index
    return {
        "Ensemble_C_best_plus_statistical": [best, best_of(["statistical"])],
        "Ensemble_D_best_plus_other_ml": [best, best_of({"lgbm_l1", "lgbm_other", "catboost"} - {family[best]})],
        "Ensemble_E_diverse": diverse,
        "Ensemble_F_family": [best] + [best_of([f]) for f in other_families],
    }


fixed = ensemble_rules(completed_runs(TEST_CUTOFF))
fixed_members = {"A_P8_top3": p8_top3, "B_best_individual": [single.idxmin()]} | fixed
pd.DataFrame({
    "members": {name: ", ".join(members) for name, members in fixed_members.items()},
    "families": {name: ", ".join(family[m] for m in members) for name, members in fixed_members.items()},
    "mean_member_correlation": {name: errors[members].corr().where(~np.eye(len(members), dtype=bool)).stack().mean() for name, members in fixed_members.items()},
})

,members,families,mean_member_correlation
A_P8_top3,"lgbm_l1_cat_price_response, lgbm_l1_cat_combined, lgbm_l1_cat_price","lgbm_l1, lgbm_l1, lgbm_l1",0.995681
B_best_individual,lgbm_l1_cat_price_response,lgbm_l1,NaN
Ensemble_C_best_plus_statistical,"lgbm_l1_cat_price_response, SeasonalNaive","lgbm_l1, statistical",0.602076
Ensemble_D_best_plus_other_ml,"lgbm_l1_cat_price_response, lgbm_promo_calendar_categorical","lgbm_l1, lgbm_other",0.915371
Ensemble_E_diverse,"lgbm_l1_cat_price_response, lgbm_l1_std, lgbm_l1_cat_static","lgbm_l1, lgbm_l1, lgbm_l1",0.944818
Ensemble_F_family,"lgbm_l1_cat_price_response, lgbm_promo_calendar_categorical, SeasonalNaive","lgbm_l1, lgbm_other, statistical",0.707116


### Fixed ensembles on the development runs

The members above, chosen with all seven development runs, scored on those same runs. This is in-sample for the choice, so it flatters every rule; it shows whether a rule is competitive at all. WAPE by development origin, the mean change against the best individual model and its standard error over the seven origins. The origins overlap by ten weeks, so the standard error is optimistic.

In [53]:
cutoff_level = errors.index.get_level_values("cutoff")
by_origin = pd.DataFrame({name: errors[members].mean(axis=1).abs().groupby(cutoff_level).sum() / y.groupby(cutoff_level).sum()
                          for name, members in fixed_members.items()}).T
change = (by_origin - by_origin.loc["B_best_individual"]) * 100
by_origin.round(3).assign(
    dev_wape=pd.Series({name: errors[m].mean(axis=1).abs().sum() / y.sum() for name, m in fixed_members.items()}).round(3),
    change_points=change.mean(axis=1).round(2),
    standard_error=(change.std(axis=1) / np.sqrt(change.shape[1])).round(2),
    beats_best=(change < 0).sum(axis=1),
)

cutoff,2024-07-17 00:00:00,2024-08-14 00:00:00,2024-09-11 00:00:00,2024-10-09 00:00:00,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,dev_wape,change_points,standard_error,beats_best
A_P8_top3,0.337,0.333,0.374,0.359,0.349,0.372,0.340,0.352,-0.03,0.02,5
B_best_individual,0.339,0.333,0.374,0.359,0.349,0.372,0.340,0.353,0.00,0.00,0
Ensemble_C_best_plus_statistical,0.368,0.361,0.367,0.348,0.337,0.341,0.340,0.351,-0.06,0.83,5
Ensemble_D_best_plus_other_ml,0.356,0.348,0.390,0.370,0.362,0.383,0.346,0.365,1.27,0.14,0
Ensemble_E_diverse,0.341,0.336,0.374,0.360,0.349,0.386,0.340,0.356,0.28,0.18,2
Ensemble_F_family,0.361,0.354,0.374,0.353,0.344,0.353,0.337,0.353,0.14,0.57,5


### Rolling, time-valid ensembles

The same rules applied at each of the six decision dates with only the runs completed by then, so earlier decisions choose from one to three runs. Scored with `leaderboard` on the final 14 weeks (exploratory) together with A and B.

In [54]:
rolling = {d: ensemble_rules(completed_runs(d)) for d in all_decisions}
ensemble_runs = pd.concat([average(name, members, [d]) for d, rules in rolling.items() for name, members in rules.items()])
benchmarks = behavior_runs[behavior_runs.model.isin(["Average_top3_ml_behavior", "Best_P8_behavior"])]
diversity_compare = pd.concat([benchmarks, ensemble_runs])

diversity_board = leaderboard(diversity_compare, actuals, scope="test").set_index("model")
diversity_board[["wape", "bias", "mean_rmsse", "median_rmsse", "apc", "coverage"]].round(3)

,wape,bias,mean_rmsse,median_rmsse,apc,coverage
model,,,,,,
Ensemble_D_best_plus_other_ml,0.341,-0.062,1.239,0.662,0.106,1.0
Best_P8_behavior,0.344,-0.035,1.249,0.664,0.114,1.0
Ensemble_F_family,0.344,-0.054,1.265,0.691,0.070,1.0
Average_top3_ml_behavior,0.345,-0.032,1.249,0.668,0.110,1.0
Ensemble_E_diverse,0.352,-0.018,1.241,0.664,0.107,1.0
Ensemble_C_best_plus_statistical,0.356,-0.036,1.304,0.733,0.057,1.0


In [55]:
by_cutoff = pd.concat([score(diversity_compare[diversity_compare.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in all_decisions])
diversity_wape = by_cutoff.pivot(index="model", columns="cutoff", values="wape")
change = (diversity_wape - diversity_wape.loc["Best_P8_behavior"]) * 100
change.drop(index="Best_P8_behavior").round(2).assign(beats_best=(change.drop(index="Best_P8_behavior") < 0).sum(axis=1))

cutoff,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-02-12 00:00:00,2025-03-12 00:00:00,2025-04-09 00:00:00,beats_best
model,,,,,,,
Average_top3_ml_behavior,0.06,-0.02,-0.20,-0.37,0.06,0.10,3
Ensemble_C_best_plus_statistical,2.72,1.39,2.25,1.45,0.96,1.22,0
Ensemble_D_best_plus_other_ml,1.26,1.09,0.43,0.72,-0.55,-0.25,2
Ensemble_E_diverse,0.04,1.39,0.16,-0.34,-0.29,0.79,2
Ensemble_F_family,2.28,1.45,1.78,0.76,-0.48,0.02,1


Members chosen at each decision date. Rules whose members change from date to date are less settled than the fixed table suggests.

In [56]:
pd.DataFrame({d.date(): {name: ", ".join(members) for name, members in rules.items()} for d, rules in sorted(rolling.items())}).T

,Ensemble_C_best_plus_statistical,Ensemble_D_best_plus_other_ml,Ensemble_E_diverse,Ensemble_F_family
2024-11-06,"lgbm_l1_cat_combined, Naive","lgbm_l1_cat_combined, lgbm_promo_calendar_categorical","lgbm_l1_cat_combined, lgbm_l1_cat_shape, lgbm_l1_std","lgbm_l1_cat_combined, lgbm_promo_calendar_categorical, Naive"
2024-12-04,"lgbm_l1_cat_price_response, HistoricAverage","lgbm_l1_cat_price_response, lgbm_promo_calendar_categorical","lgbm_l1_cat_price_response, lgbm_l1_std, lgbm_l1_cat_distribution","lgbm_l1_cat_price_response, lgbm_promo_calendar_categorical, HistoricAverage"
2025-01-01,"lgbm_l1_cat_price, HistoricAverage","lgbm_l1_cat_price, lgbm_promo_calendar_categorical","lgbm_l1_cat_price, lgbm_l1_std, lgbm_l1_cat_shape","lgbm_l1_cat_price, lgbm_promo_calendar_categorical, HistoricAverage"
2025-02-12,"lgbm_l1_cat_price_response, SeasonalNaive","lgbm_l1_cat_price_response, lgbm_promo_calendar_none","lgbm_l1_cat_price_response, lgbm_l1_std, lgbm_l1_cat_static","lgbm_l1_cat_price_response, lgbm_promo_calendar_none, SeasonalNaive"
2025-03-12,"lgbm_l1_cat_price_response, SeasonalNaive","lgbm_l1_cat_price_response, lgbm_promo_calendar_categorical","lgbm_l1_cat_price_response, lgbm_l1_std, lgbm_l1_cat_static","lgbm_l1_cat_price_response, lgbm_promo_calendar_categorical, SeasonalNaive"
2025-04-09,"lgbm_l1_cat_price_response, SeasonalNaive","lgbm_l1_cat_price_response, lgbm_promo_calendar_categorical","lgbm_l1_cat_price_response, lgbm_l1_std, lgbm_l1_cat_static","lgbm_l1_cat_price_response, lgbm_promo_calendar_categorical, SeasonalNaive"


### Reading the diversity results

- **Near-duplicates are real.** The three P8 top-three members have error correlations of 0.995 to 0.997, so their average is the best model with noise. The most different competitive errors come from seasonal naive (correlation about 0.60 with the best LightGBM, opposite sign in a third of product-weeks), then CatBoost (0.82), then the other LightGBM models (0.91 to 0.92).
- **Development evidence supports none of the diverse ensembles over the best model.** On the seven development runs that chose them, C (best plus seasonal naive) and F (best, `lgbm_promo_calendar_categorical`, seasonal naive) are level with the best individual on average, but with large swings by origin: C changes by -3.1 to +2.9 points. D and E are worse (+1.3 and +0.3 points). The accuracy-selected A is level too, within 0.2 points at every origin.
- **Time-valid rolling selection is worse still.** Applied at each decision date with only the runs completed by then, C is worse than the best individual at all six dates, F at five, D and E at four. A is within 0.4 points at every date.
- **The final window does not change the conclusion.** D scores 0.341 against 0.344 for the best individual, but development rejected it at every origin, so that is not evidence for D. F ties the best individual (0.344), C and E are worse (0.356 and 0.352).
- **Diversity buys stability, not accuracy.** Ensembles with seasonal naive revise far less between origins: final-window APC 0.057 for C and 0.070 for F, against 0.114 for the best individual. F gets that at equal WAPE, with a slightly more negative bias (-0.054 against -0.035).

On this evidence the best model on its own is the supported choice. Accuracy-based selection only adds near-copies of it, explicit residual diversity picks weaker members, and simple family diversity is the only rule that costs no accuracy, while making the forecasts noticeably more stable. If stability matters, F is the ensemble to consider; it is not more accurate.

In [57]:
save_forecasts(pd.concat([strategies, hier_runs[hier_runs.cutoff.isin(targets)], tour_runs[tour_runs.cutoff.isin(targets)], pool_averages[pool_averages.cutoff.isin(targets)], hybrid_runs[hybrid_runs.cutoff.isin(targets)], catboost_runs[catboost_runs.cutoff.isin(targets) & ~catboost_runs.model.str.startswith("Best_")],
                        categorical_runs[categorical_runs.cutoff.isin(targets) & ~categorical_runs.model.str.startswith("Best_")],
                        behavior_runs[behavior_runs.cutoff.isin(targets) & ~behavior_runs.model.str.startswith("Best_")], planned_runs, hindsight]), "07_model_selection")